# Smart Energy Consumption Analytics System

This notebook is the main technical workflow for the Data Mining final project. It follows the CRISP-DM logic from the report, but keeps the code practical: load the household power dataset, clean it, engineer time and energy features, explore usage patterns, then run baseline mining models.

The project goal is not only to get model scores. The goal is to explain household electricity behavior in a way that can support energy-saving decisions, anomaly detection, and a dashboard demo.

## How We Will Run This Notebook

The raw dataset has more than 2 million minute-level rows. Your PC can handle it, but during development it is smarter to run a large sample first, verify the code, then switch to the full dataset for final numbers.

- `USE_FULL_DATA = False` means fast development mode.
- `USE_FULL_DATA = True` means final full-dataset run.
- `RESAMPLE_FREQ = "h"` converts minute data into hourly data, which is better for most models and dashboard charts.

This keeps the workflow controlled instead of turning every small change into a long wait.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

# Make the local src/ package importable when the notebook is opened from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_preparation import (
    DEFAULT_DATA_PATH,
    available_forecast_features,
    available_model_features,
    prepare_energy_dataset,
    reduce_memory_usage,
    summarize_dataset,
)
from src.modeling import (
    detect_anomalies,
    pca_feature_summary,
    run_clustering_baselines,
    summarize_clusters,
    train_classification_baselines,
    train_regression_baselines,
)

sns.set_theme(style="whitegrid")

In [ ]:
# Development control panel.
# Start with sample mode. For final results, set USE_FULL_DATA to True and rerun all cells.
USE_FULL_DATA = False
SAMPLE_ROWS = 300_000
RESAMPLE_FREQ = "h"

DATA_PATH = DEFAULT_DATA_PATH
NROWS = None if USE_FULL_DATA else SAMPLE_ROWS

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Mode: {'full dataset' if USE_FULL_DATA else f'sample of {SAMPLE_ROWS:,} rows'}")

## 1. Data Understanding and Preparation

The raw file stores `Date` and `Time` separately and stores some missing values as `?`. We combine date/time into one timestamp, convert numeric columns, interpolate small time-series gaps, and create features that are easier to mine.

Important engineered features:

- `hour`, `day_of_week`, `month`, `is_weekend`: help detect time-based consumption behavior.
- `sub_metering_total_wh`: total measured appliance-category consumption.
- `active_energy_wh`: estimated total energy per time period.
- `unmetered_energy_wh`: energy not explained by the three sub-metering columns.
- `high_consumption`: classification target based on the upper quartile of power usage.

In [ ]:
energy_df = prepare_energy_dataset(
    data_path=DATA_PATH,
    nrows=NROWS,
    frequency=RESAMPLE_FREQ,
)
energy_df = reduce_memory_usage(energy_df)

summary = summarize_dataset(energy_df)
print(
    "Working CSV coverage:",
    summary["start"].date(),
    "to",
    summary["end"].date(),
)
summary

In [ ]:
# A quick inspection cell. If this looks wrong, we stop here before modeling.
display(energy_df.head())
display(energy_df.describe().T)

## 2. Exploratory Data Analysis

This section answers the first business question: when does household electricity usage rise or fall? The dashboard will later reuse the same ideas with filters and interactive charts.

In [ ]:
fig = px.line(
    energy_df,
    x="datetime",
    y="global_active_power",
    title="Global Active Power Over Time",
    labels={"global_active_power": "Global active power (kW)", "datetime": "Date"},
)
fig.show()

In [ ]:
hourly_profile = (
    energy_df.groupby("hour", as_index=False)["global_active_power"]
    .mean()
)

fig = px.bar(
    hourly_profile,
    x="hour",
    y="global_active_power",
    title="Average Consumption by Hour of Day",
    labels={"global_active_power": "Average power (kW)", "hour": "Hour"},
)
fig.show()

In [ ]:
metering_cols = ["sub_metering_1", "sub_metering_2", "sub_metering_3", "unmetered_energy_wh"]
metering_totals = energy_df[metering_cols].sum().reset_index()
metering_totals.columns = ["source", "energy_wh"]

fig = px.pie(
    metering_totals,
    names="source",
    values="energy_wh",
    title="Estimated Energy Share by Metering Source",
)
fig.show()

## 3. Modeling Feature Set

The same feature list is shared across the first baseline models. This keeps the project easier to explain: the models are seeing the same prepared household behavior signals, then each mining technique uses them differently.

In [ ]:
feature_columns = available_model_features(energy_df)
forecast_feature_columns = available_forecast_features(energy_df)

print("Behavior/mining features:")
print(feature_columns)
print("\nForecasting features:")
print(forecast_feature_columns)

## 4. Clustering: Usage Behavior Segments

Clustering groups similar time periods together. In the final interpretation, clusters can be described as normal low load, active household periods, and high-consumption periods if the profiles support that reading.

For visualization, we project the multi-feature clustering space into two PCA components. This is better than plotting `global_active_power` against `global_intensity`, because those two columns are almost perfectly correlated and mostly create a diagonal line.

In [ ]:
clustering_metrics, clustered_df = run_clustering_baselines(
    energy_df,
    features=feature_columns,
    max_rows=20_000,
)
display(clustering_metrics)

cluster_profile = summarize_clusters(clustered_df)
display(cluster_profile)

fig = px.scatter(
    clustered_df,
    x="pca_1",
    y="pca_2",
    color="kmeans_cluster",
    hover_data=["datetime", "global_active_power", "global_intensity", "sub_metering_total_wh"],
    title="K-Means Clusters Projected with PCA",
    labels={"pca_1": "PCA component 1", "pca_2": "PCA component 2"},
)
fig.show()

## 5. Regression: Predicting Consumption

Regression estimates future or current active power from time and household usage features. We use a time-aware split by setting `shuffle=False`, so the model trains on earlier records and tests on later records.

In [ ]:
regression_metrics, regression_models = train_regression_baselines(
    energy_df,
    features=forecast_feature_columns,
    target="global_active_power",
)
display(regression_metrics.sort_values("rmse"))

## 6. Classification: High vs Normal Consumption

Classification turns the problem into a decision question: is this period high consumption or not? This is useful for alerting and dashboard labeling.

In [ ]:
classification_metrics, classification_models, classification_report_text = train_classification_baselines(
    energy_df,
    features=feature_columns,
    target="high_consumption",
)
display(classification_metrics.sort_values("accuracy", ascending=False))
print(classification_report_text)

## 7. Anomaly Detection: Unusual Consumption Events

Anomaly detection searches for periods that do not look like normal household behavior. These may represent appliance spikes, data issues, or unusual usage patterns worth investigating.

In [ ]:
anomaly_df = detect_anomalies(
    energy_df,
    features=feature_columns,
    contamination=0.01,
    max_rows=100_000,
)

display(anomaly_df["anomaly_label"].value_counts().rename("count"))

fig = px.scatter(
    anomaly_df,
    x="datetime",
    y="global_active_power",
    color="anomaly_label",
    title="Detected Anomalies in Household Consumption",
    labels={"global_active_power": "Global active power (kW)", "anomaly_label": "Anomaly"},
)
fig.show()

## 8. PCA: Structure in the Feature Space

PCA helps explain how much of the variation in household behavior can be summarized by a smaller number of components. It is useful for anomaly detection discussion and dimensionality reduction.

In [ ]:
pca_summary, pca_model = pca_feature_summary(
    energy_df,
    features=feature_columns,
    n_components=3,
)
display(pca_summary)

fig = px.bar(
    pca_summary,
    x="component",
    y="explained_variance_ratio",
    title="PCA Explained Variance by Component",
)
fig.show()

## 9. Save Prepared Data for the Dashboard

The dashboard should not repeat expensive cleaning work every time if we already prepared a clean hourly file. This cell creates a reusable CSV in `outputs/`.

In [ ]:
output_path = PROJECT_ROOT / "outputs" / "prepared_hourly_energy.csv"
energy_df.to_csv(output_path, index=False)
print(f"Saved prepared data to: {output_path}")